# Prediksi Rating Review Game Wuthering Waves

## Latar Belakang

Proyek ini bertujuan untuk membangun model prediksi rating berdasarkan teks review game Wuthering Waves yang dikumpulkan dari Google Play Store. Model ini dapat digunakan oleh pihak pengembang game untuk memprediksi sentimen dari berbagai sumber konten.

## Alur Kerja Proyek

1. Scraping data review
2. Exploratory Data Analysis (EDA)
3. Preprocessing data teks
4. Text Representation (TF-IDF dan Word2Vec)
5. Pemodelan Machine Learning
6. Pemodelan Deep Learning
7. Evaluasi performa model
8. Penanganan imbalance data


## Instalasi Library yang Diperlukan

In [ ]:
# Instalasi semua library yang dibutuhkan dalam proyek ini
!pip install google-play-scraper
!pip install gensim
!pip install imbalanced-learn
!pip install wordcloud
!pip install nltk
!pip install scikit-learn
!pip install tensorflow
!pip install pandas numpy matplotlib seaborn

## Import Semua Library

In [ ]:
# ================================================================
# IMPORT SEMUA LIBRARY YANG DIGUNAKAN DALAM PROYEK
# ================================================================

# Library untuk scraping data dari Google Play Store
from google_play_scraper import reviews, Sort

# Library untuk manipulasi dan analisis data
import pandas as pd
import numpy as np

# Library untuk visualisasi data
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud

# Library untuk pemrosesan teks NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
import re
import string

# Library untuk representasi teks
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

# Library untuk Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Library untuk penanganan imbalance data
from imblearn.over_sampling import SMOTE

# Library untuk Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, Dropout, Bidirectional, Input
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Library pendukung lainnya
import warnings
import time
import json

# Menonaktifkan peringatan yang tidak diperlukan agar output lebih bersih
warnings.filterwarnings('ignore')

# Mengunduh resource NLTK yang dibutuhkan untuk tokenisasi dan lemmatization
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

# Menetapkan seed global agar hasil eksperimen dapat direproduksi
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Semua library berhasil diimpor.")

---
# Bagian 1 - Scraping Data Review

## Tujuan

Pada bagian ini, kita mengumpulkan data review game Wuthering Waves dari Google Play Store menggunakan library `google-play-scraper`. Data yang dikumpulkan akan menjadi bahan utama untuk melatih model prediksi rating.

## Library yang Digunakan

Library `google-play-scraper` dipilih karena:
- Mudah digunakan dan tidak memerlukan API key
- Mendukung filter bahasa sehingga kita dapat mengambil review berbahasa Inggris saja
- Mendukung pengambilan data dalam jumlah besar secara bertahap

## Strategi Scraping

- Target: minimal 2000 data review
- Filter bahasa: Inggris (`lang='en'`)
- Data diambil dari berbagai urutan (terbaru dan paling relevan) untuk memastikan variasi rating
- App ID Wuthering Waves di Google Play: `com.kurogame.wutheringwaves.global`


In [ ]:
# ================================================================
# BAGIAN 1: SCRAPING DATA REVIEW WUTHERING WAVES
# ================================================================

# ID aplikasi Wuthering Waves di Google Play Store
APP_ID = 'com.kurogame.wutheringwaves.global'

def scrape_reviews(app_id, target_count=2000):
    """
    Fungsi untuk mengumpulkan review dari Google Play Store.
    
    Parameter:
    - app_id: ID aplikasi di Google Play Store
    - target_count: jumlah review yang ingin dikumpulkan
    
    Mengembalikan DataFrame berisi data review.
    """
    all_reviews = []
    
    # Mengambil review dari dua jenis pengurutan untuk mendapatkan variasi rating
    # Sort.NEWEST mengambil review terbaru
    # Sort.MOST_RELEVANT mengambil review yang dianggap paling relevan oleh platform
    sort_methods = [Sort.NEWEST, Sort.MOST_RELEVANT]
    count_per_sort = target_count // len(sort_methods) + 200

    for sort_method in sort_methods:
        print(f"Mengambil review dengan metode pengurutan: {sort_method.name}...")
        continuation_token = None
        batch_count = 0
        
        # Mengambil review secara bertahap (per 200 data) hingga mencapai target
        while batch_count < count_per_sort:
            try:
                # Memanggil fungsi reviews dari google-play-scraper
                result, continuation_token = reviews(
                    app_id,
                    lang='en',           # Hanya review berbahasa Inggris
                    country='us',        # Dari region Amerika Serikat
                    sort=sort_method,    # Metode pengurutan
                    count=200,           # Jumlah review per permintaan
                    continuation_token=continuation_token
                )
                
                # Menghentikan proses jika tidak ada data lagi
                if not result:
                    print("Tidak ada data lagi yang tersedia.")
                    break
                
                # Menambahkan hasil ke daftar semua review
                all_reviews.extend(result)
                batch_count += len(result)
                print(f"  Total review terkumpul: {len(all_reviews)}")
                
                # Memberi jeda 1 detik antar permintaan agar tidak diblokir server
                time.sleep(1)
                
                # Menghentikan jika sudah melebihi target
                if len(all_reviews) >= target_count * 1.2:
                    break
                    
            except Exception as e:
                print(f"Terjadi error: {e}")
                time.sleep(3)
                break
    
    return all_reviews

# Menjalankan proses scraping
print("Memulai proses scraping data review Wuthering Waves...")
raw_reviews = scrape_reviews(APP_ID, target_count=2000)
print(f"\nTotal review berhasil dikumpulkan: {len(raw_reviews)}")

In [ ]:
# Mengubah data hasil scraping menjadi DataFrame pandas
df_raw = pd.DataFrame(raw_reviews)

# Memilih kolom yang relevan untuk analisis
# 'content' adalah teks review, 'score' adalah rating (1-5), 'at' adalah tanggal
df = df_raw[['content', 'score', 'at']].copy()

# Mengubah nama kolom agar lebih mudah dipahami
df.columns = ['review', 'rating', 'date']

# Menghapus baris yang memiliki nilai kosong
df.dropna(subset=['review', 'rating'], inplace=True)

# Menghapus duplikat review yang mungkin muncul akibat tumpang tindih metode pengurutan
df.drop_duplicates(subset=['review'], inplace=True)

# Mengonversi kolom rating menjadi tipe integer
df['rating'] = df['rating'].astype(int)

# Mengambil tepat 2000 data jika lebih
if len(df) > 2000:
    df = df.sample(n=2000, random_state=SEED).reset_index(drop=True)

# Menyimpan data ke file CSV untuk penggunaan selanjutnya
df.to_csv('wuthering_waves_reviews.csv', index=False)
print("Data berhasil disimpan ke file 'wuthering_waves_reviews.csv'")

# Menampilkan informasi dasar tentang dataset
print(f"\nJumlah data: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"\nTipe data setiap kolom:")
print(df.dtypes)
print(f"\nJumlah nilai kosong per kolom:")
print(df.isnull().sum())

# Menampilkan 5 baris pertama data
print("\n5 Baris Pertama Data:")
df.head()

---
# Bagian 2 - Exploratory Data Analysis (EDA)

## Tujuan

Pada bagian ini, kita melakukan eksplorasi terhadap data yang telah dikumpulkan untuk memahami karakteristiknya sebelum melakukan pemrosesan lebih lanjut. EDA membantu kita:

- Memahami distribusi rating dalam dataset
- Mengidentifikasi kata-kata yang paling sering muncul di setiap rating
- Mendeteksi kata-kata tidak baku yang memiliki frekuensi tinggi
- Memahami panjang rata-rata review per rating


In [ ]:
# ================================================================
# BAGIAN 2: EXPLORATORY DATA ANALYSIS
# ================================================================

# Memuat ulang data dari file CSV (memastikan konsistensi data)
df = pd.read_csv('wuthering_waves_reviews.csv')

# --- 2.1 Statistik Deskriptif ---
print("=" * 50)
print("STATISTIK DESKRIPTIF DATASET")
print("=" * 50)
print(df.describe())

print("\nDistribusi Rating:")
print(df['rating'].value_counts().sort_index())

In [ ]:
# --- 2.2 Visualisasi Distribusi Rating ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Menghitung jumlah data per rating
rating_counts = df['rating'].value_counts().sort_index()

# Bar chart distribusi rating
colors = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1976d2']
axes[0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribusi Rating Review Wuthering Waves', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating (1-5 Bintang)')
axes[0].set_ylabel('Jumlah Review')
axes[0].set_xticks([1, 2, 3, 4, 5])
for i, v in enumerate(rating_counts.values):
    axes[0].text(i + 1, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart distribusi rating
axes[1].pie(
    rating_counts.values,
    labels=[f'Rating {i}' for i in rating_counts.index],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Proporsi Rating Review', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('distribusi_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualisasi distribusi rating berhasil dibuat.")

In [ ]:
# --- 2.3 WordCloud untuk Setiap Rating ---
# WordCloud memvisualisasikan kata yang sering muncul, semakin besar ukuran kata
# semakin sering kata tersebut muncul dalam data

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

wc_colors = {
    1: 'Reds', 2: 'Oranges', 3: 'YlOrBr',
    4: 'Greens', 5: 'Blues'
}

for i, rating in enumerate([1, 2, 3, 4, 5]):
    # Menggabungkan semua teks review untuk rating tertentu menjadi satu string
    text_for_rating = ' '.join(
        df[df['rating'] == rating]['review'].dropna().astype(str).tolist()
    )
    
    # Membuat objek WordCloud
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap=wc_colors[rating],
        max_words=100,
        collocations=False
    ).generate(text_for_rating)
    
    # Menampilkan WordCloud pada subplot
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f'WordCloud Rating {rating} Bintang', fontsize=12, fontweight='bold')
    axes[i].axis('off')

# Menyembunyikan subplot yang tidak digunakan (indeks ke-5)
axes[5].axis('off')

plt.suptitle('WordCloud Review Wuthering Waves per Rating', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('wordcloud_per_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print("WordCloud per rating berhasil dibuat.")

In [ ]:
# --- 2.4 Frekuensi Kata Paling Umum per Rating ---
# Fungsi untuk menghitung kata paling sering muncul
def get_top_words(text_series, top_n=15):
    """
    Menghitung kata yang paling sering muncul dalam kumpulan teks.
    
    Parameter:
    - text_series: Series pandas berisi teks review
    - top_n: jumlah kata teratas yang ingin ditampilkan
    """
    # Menggabungkan semua teks dan memecahnya menjadi daftar kata
    all_words = ' '.join(text_series.dropna().astype(str)).lower().split()
    # Menghitung frekuensi setiap kata
    word_freq = Counter(all_words)
    return word_freq.most_common(top_n)

# Membuat visualisasi frekuensi kata untuk setiap rating
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

bar_colors = ['#ef5350', '#ff7043', '#ffca28', '#66bb6a', '#42a5f5']

for i, rating in enumerate([1, 2, 3, 4, 5]):
    # Mendapatkan kata paling umum untuk rating ini
    top_words = get_top_words(df[df['rating'] == rating]['review'], top_n=15)
    words = [w[0] for w in top_words]
    freqs = [w[1] for w in top_words]
    
    # Membuat bar chart horizontal
    axes[i].barh(words[::-1], freqs[::-1], color=bar_colors[i], edgecolor='gray', linewidth=0.3)
    axes[i].set_title(f'15 Kata Paling Umum - Rating {rating} Bintang', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Frekuensi')

axes[5].axis('off')
plt.suptitle('Kata Paling Sering Muncul per Rating (Sebelum Preprocessing)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('frekuensi_kata_per_rating.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 2.5 Deteksi Kata Tidak Baku (Slang dan Singkatan) ---
# Daftar kata tidak baku yang umum ditemukan dalam review game
slang_words = [
    'ur', 'u', 'pls', 'plz', 'rly', 'imo', 'tbh', 'lol', 'omg', 'wtf',
    'ngl', 'idk', 'gg', 'op', 'af', 'fr', 'bruh', 'bro', 'sis', 'gonna',
    'wanna', 'gotta', 'coz', 'cuz', 'tho', 'thru', 'b4', 'w/', 'w/o',
    'bc', 'rn', 'atm', 'asap', 'smh', 'fyi', 'btw', 'dm', 'acc',
    '2', '4', 'gr8', 'l8r', 'nah', 'yea', 'yep', 'yup'
]

# Menggabungkan semua teks review menjadi satu dan memisahnya per kata
all_words_full = ' '.join(df['review'].dropna().astype(str)).lower().split()
word_counter = Counter(all_words_full)

# Menghitung kemunculan kata-kata tidak baku dalam dataset
slang_freq = {word: word_counter.get(word, 0) for word in slang_words}
slang_df = pd.DataFrame(list(slang_freq.items()), columns=['Kata Tidak Baku', 'Frekuensi'])
slang_df = slang_df[slang_df['Frekuensi'] > 0].sort_values('Frekuensi', ascending=False)

print("Kata tidak baku yang ditemukan dalam dataset:")
print(slang_df.to_string(index=False))

# Visualisasi frekuensi kata tidak baku
if not slang_df.empty:
    plt.figure(figsize=(12, 5))
    plt.bar(slang_df['Kata Tidak Baku'], slang_df['Frekuensi'], color='#ab47bc', edgecolor='black', linewidth=0.4)
    plt.title('Frekuensi Kata Tidak Baku dalam Dataset Review', fontsize=13, fontweight='bold')
    plt.xlabel('Kata Tidak Baku')
    plt.ylabel('Frekuensi Kemunculan')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('kata_tidak_baku.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- 2.6 Analisis Panjang Review per Rating ---
# Menghitung jumlah kata dalam setiap review
df['review_length'] = df['review'].astype(str).apply(lambda x: len(x.split()))

# Menghitung rata-rata panjang review per rating
avg_length = df.groupby('rating')['review_length'].mean()

plt.figure(figsize=(10, 5))
plt.bar(avg_length.index, avg_length.values, color=colors, edgecolor='black', linewidth=0.5)
plt.title('Rata-rata Panjang Review (Jumlah Kata) per Rating', fontsize=13, fontweight='bold')
plt.xlabel('Rating (Bintang)')
plt.ylabel('Rata-rata Jumlah Kata')
plt.xticks([1, 2, 3, 4, 5])
for i, (idx, val) in enumerate(avg_length.items()):
    plt.text(idx, val + 0.5, f'{val:.1f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('panjang_review_per_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print("Rata-rata panjang review per rating:")
print(avg_length)

---
# Bagian 3 - Preprocessing Data

## Tujuan

Data teks mentah tidak dapat langsung digunakan sebagai input model machine learning. Tahap preprocessing bertujuan untuk membersihkan dan menormalkan teks sehingga menghasilkan representasi yang lebih bersih, konsisten, dan bermakna.

## Langkah-langkah Preprocessing

1. **Case Folding**: Mengubah semua teks menjadi huruf kecil untuk menghindari duplikasi kata yang sama namun berbeda kapitalisasi (contoh: "Game" dan "game" dianggap sebagai kata yang sama).

2. **Pembersihan Teks**: Menghapus elemen yang tidak membawa makna linguistik seperti URL, angka, tanda baca, dan karakter khusus.

3. **Tokenisasi**: Memecah kalimat menjadi token (kata-kata individual) menggunakan NLTK.

4. **Penghapusan Stopwords**: Menghapus kata-kata yang sangat umum dan tidak membawa makna signifikan (contoh: "the", "is", "at").

5. **Normalisasi Slang**: Mengganti kata tidak baku dengan kata standar menggunakan kamus slang.

6. **Lemmatization**: Mengubah kata ke bentuk dasarnya (lemma) dengan mempertimbangkan kelas kata. Lemmatization dipilih dibanding stemming karena:
   - Lemmatization menghasilkan kata yang valid secara linguistik (contoh: "running" -> "run", bukan "runn" seperti pada stemming)
   - Kelas kata dipertahankan sehingga makna tidak berubah
   - Cocok untuk analisis yang memerlukan representasi kata yang bermakna


In [ ]:
# ================================================================
# BAGIAN 3: PREPROCESSING DATA
# ================================================================

# Memuat data
df = pd.read_csv('wuthering_waves_reviews.csv')

# Inisialisasi tools NLTK yang akan digunakan
lemmatizer = WordNetLemmatizer()  # Untuk mengubah kata ke bentuk dasarnya
stop_words = set(stopwords.words('english'))  # Kumpulan stopwords bahasa Inggris

# Kamus normalisasi kata tidak baku (slang) ke kata standar bahasa Inggris
# Kamus ini berisi kata-kata yang umum ditemukan dalam review game
SLANG_DICT = {
    'ur': 'your', 'u': 'you', 'pls': 'please', 'plz': 'please',
    'rly': 'really', 'imo': 'in my opinion', 'tbh': 'to be honest',
    'lol': 'laugh', 'omg': 'oh my god', 'ngl': 'not gonna lie',
    'idk': 'i do not know', 'gg': 'good game', 'op': 'overpowered',
    'af': 'very', 'fr': 'for real', 'bruh': 'bro', 'gonna': 'going to',
    'wanna': 'want to', 'gotta': 'got to', 'coz': 'because',
    'cuz': 'because', 'tho': 'though', 'thru': 'through',
    'b4': 'before', 'bc': 'because', 'rn': 'right now',
    'atm': 'at the moment', 'asap': 'as soon as possible',
    'smh': 'shaking my head', 'fyi': 'for your information',
    'btw': 'by the way', 'nah': 'no', 'yea': 'yes', 'yep': 'yes',
    'yup': 'yes', 'gr8': 'great', 'l8r': 'later', 'acc': 'actually',
    'wtf': 'what the heck', 'w/': 'with', 'w/o': 'without'
}

def normalize_slang(text, slang_dict):
    """
    Mengganti kata-kata tidak baku dengan kata standar menggunakan kamus slang.
    
    Parameter:
    - text: string teks yang akan dinormalisasi
    - slang_dict: kamus berisi pasangan {kata_tidak_baku: kata_standar}
    """
    words = text.split()
    # Mengganti setiap kata jika ada dalam kamus slang
    normalized = [slang_dict.get(word, word) for word in words]
    return ' '.join(normalized)

def preprocess_text(text):
    """
    Fungsi utama preprocessing teks yang menggabungkan semua langkah pembersihan.
    
    Langkah:
    1. Case folding (huruf kecil)
    2. Pembersihan karakter tidak perlu
    3. Normalisasi slang
    4. Tokenisasi
    5. Penghapusan stopwords
    6. Lemmatization
    """
    # Mengabaikan input yang bukan string
    if not isinstance(text, str):
        return ''
    
    # Langkah 1: Case folding - mengubah semua huruf menjadi kecil
    text = text.lower()
    
    # Langkah 2: Pembersihan teks
    # Menghapus URL (http/https/www)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Menghapus angka
    text = re.sub(r'\d+', '', text)
    # Menghapus karakter khusus dan tanda baca
    text = re.sub(r'[^\w\s]', ' ', text)
    # Menghapus underscore
    text = re.sub(r'_', ' ', text)
    # Menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Langkah 3: Normalisasi kata tidak baku (slang)
    text = normalize_slang(text, SLANG_DICT)
    
    # Langkah 4: Tokenisasi - memecah teks menjadi daftar kata
    tokens = word_tokenize(text)
    
    # Langkah 5: Penghapusan stopwords dan token yang terlalu pendek
    tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    
    # Langkah 6: Lemmatization - mengubah kata ke bentuk dasar (lemma)
    # Lemmatization mempertahankan kelas kata sehingga makna tidak berubah
    tokens = [lemmatizer.lemmatize(token, pos='v') for token in tokens]
    # Lemmatization kedua kali untuk memastikan kata benda juga diproses dengan benar
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Menggabungkan kembali token menjadi string
    return ' '.join(tokens)

print("Memulai proses preprocessing teks...")
print("Ini mungkin memerlukan beberapa menit.")

# Menerapkan fungsi preprocessing ke seluruh kolom review
df['review_clean'] = df['review'].apply(preprocess_text)

# Menghapus baris dengan hasil preprocessing yang kosong
df = df[df['review_clean'].str.strip() != ''].reset_index(drop=True)

print(f"Preprocessing selesai. Jumlah data tersisa: {len(df)}")

In [ ]:
# --- Menampilkan Perbandingan Sebelum dan Sesudah Preprocessing ---
print("Perbandingan Teks Sebelum dan Sesudah Preprocessing:")
print("=" * 80)

# Mengambil 5 contoh acak untuk ditampilkan sebagai perbandingan
sample_indices = df.sample(5, random_state=SEED).index

for idx in sample_indices:
    print(f"\nContoh ke-{idx + 1}:")
    print(f"  Rating  : {df.loc[idx, 'rating']} bintang")
    print(f"  ASLI    : {df.loc[idx, 'review'][:150]}...")
    print(f"  BERSIH  : {df.loc[idx, 'review_clean'][:150]}...")
    print("-" * 80)

# Menyimpan data yang sudah dipreprocess
df.to_csv('wuthering_waves_preprocessed.csv', index=False)
print("\nData hasil preprocessing disimpan ke 'wuthering_waves_preprocessed.csv'")

---
# Bagian 4 - Text Representation

## Tujuan

Model machine learning tidak dapat memproses teks secara langsung. Kita perlu mengubah teks menjadi representasi numerik (vektor). Pada bagian ini, kita mengimplementasikan dua metode representasi teks:

## Metode 1: TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF adalah metode statistik yang mengukur **seberapa penting** sebuah kata dalam suatu dokumen relatif terhadap seluruh koleksi dokumen. Nilainya tinggi jika kata sering muncul dalam satu dokumen tetapi jarang muncul di dokumen lain.

- **TF (Term Frequency)**: Frekuensi kemunculan kata dalam satu dokumen
- **IDF (Inverse Document Frequency)**: Mengukur seberapa unik kata tersebut di seluruh korpus

## Metode 2: Word2Vec (Skip-Gram)

Word2Vec adalah metode representasi kata berbasis jaringan saraf tiruan (ANN) yang mempelajari representasi vektor kata berdasarkan konteks kata di sekitarnya. Pendekatan Skip-Gram melatih model untuk memprediksi kata-kata konteks dari sebuah kata input, sehingga vektor setiap kata merepresentasikan makna semantiknya berdasarkan kata-kata yang mengitarinya.


In [ ]:
# ================================================================
# BAGIAN 4: TEXT REPRESENTATION
# ================================================================

# Memuat data hasil preprocessing
df = pd.read_csv('wuthering_waves_preprocessed.csv')

# Memisahkan fitur (teks review) dan label (rating)
X_text = df['review_clean'].fillna('')
y = df['rating'].values

# Membagi data menjadi train set (80%) dan test set (20%)
# stratify=y memastikan proporsi kelas yang sama di kedua set
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Ukuran data training: {len(X_train_text)}")
print(f"Ukuran data testing : {len(X_test_text)}")

# ================================================================
# METODE 1: TF-IDF
# ================================================================
print("\n--- Membuat representasi TF-IDF ---")

# Menginisialisasi TF-IDF Vectorizer
# max_features=5000: hanya menggunakan 5000 kata dengan TF-IDF tertinggi
# ngram_range=(1,2): menggunakan unigram dan bigram
# min_df=2: mengabaikan kata yang muncul kurang dari 2 dokumen
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

# Melatih TF-IDF pada data training dan mengubah data training
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
# Mengubah data testing menggunakan vocabulary dari data training
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"Dimensi matriks TF-IDF training: {X_train_tfidf.shape}")
print(f"Dimensi matriks TF-IDF testing : {X_test_tfidf.shape}")

# Menampilkan contoh matriks TF-IDF (5 baris, 10 kolom pertama)
feature_names = tfidf_vectorizer.get_feature_names_out()
sample_tfidf = pd.DataFrame(
    X_train_tfidf[:5].toarray(),
    columns=feature_names
).iloc[:, :10]

print("\nContoh Matriks TF-IDF (5 baris, 10 fitur pertama):")
print(sample_tfidf.round(4))

In [ ]:
# ================================================================
# METODE 2: WORD2VEC (SKIP-GRAM)
# ================================================================
print("--- Melatih model Word2Vec ---")

# Mengubah teks menjadi daftar token untuk setiap review
# Word2Vec membutuhkan input berupa list of list of words
X_train_tokens = [text.split() for text in X_train_text]
X_test_tokens = [text.split() for text in X_test_text]

# Melatih model Word2Vec dengan pendekatan Skip-Gram (sg=1)
# Skip-Gram memprediksi kata konteks dari kata input,
# sehingga representasi vektor dipengaruhi oleh kata-kata di sekitarnya
w2v_model = Word2Vec(
    sentences=X_train_tokens,  # Data training dalam bentuk list of token
    vector_size=100,           # Dimensi vektor setiap kata
    window=5,                  # Jumlah kata konteks di kiri dan kanan
    min_count=2,               # Mengabaikan kata yang muncul kurang dari 2 kali
    workers=4,                 # Jumlah thread untuk paralelisasi
    sg=1,                      # sg=1 berarti menggunakan pendekatan Skip-Gram
    epochs=10,                 # Jumlah iterasi pelatihan
    seed=SEED
)

print(f"Model Word2Vec berhasil dilatih.")
print(f"Ukuran vocabulary: {len(w2v_model.wv)}")

# Menampilkan contoh vektor untuk beberapa kata yang relevan dengan game
game_words = ['game', 'graphic', 'character']
print("\nContoh vektor Word2Vec (5 nilai pertama dari 100 dimensi):")
for word in game_words:
    if word in w2v_model.wv:
        print(f"  '{word}': {w2v_model.wv[word][:5].round(4)}...")

# Menampilkan kata-kata yang paling mirip dengan 'game'
if 'game' in w2v_model.wv:
    print("\n5 Kata yang paling mirip dengan 'game' menurut model Word2Vec:")
    similar_words = w2v_model.wv.most_similar('game', topn=5)
    for word, score in similar_words:
        print(f"  {word}: {score:.4f}")

In [ ]:
# Fungsi untuk mengubah dokumen teks menjadi vektor Word2Vec
# menggunakan rata-rata vektor semua kata dalam dokumen tersebut
def document_vector(tokens, model, vector_size=100):
    """
    Mengubah daftar token menjadi satu vektor representasi dokumen
    dengan merata-ratakan vektor semua kata yang dikenali model.
    
    Parameter:
    - tokens: daftar kata dalam dokumen
    - model: model Word2Vec yang sudah dilatih
    - vector_size: dimensi vektor output
    """
    # Menyaring hanya kata-kata yang ada dalam vocabulary model
    valid_tokens = [token for token in tokens if token in model.wv]
    
    if not valid_tokens:
        # Mengembalikan vektor nol jika tidak ada kata yang dikenali
        return np.zeros(vector_size)
    
    # Merata-ratakan semua vektor kata yang valid
    return np.mean([model.wv[token] for token in valid_tokens], axis=0)

# Mengubah seluruh dataset menjadi matriks vektor Word2Vec
print("Mengubah data menjadi vektor Word2Vec...")
X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_test_tokens])

print(f"Dimensi matriks Word2Vec training: {X_train_w2v.shape}")
print(f"Dimensi matriks Word2Vec testing : {X_test_w2v.shape}")

# Menyimpan model Word2Vec untuk digunakan pada bagian Deep Learning
w2v_model.save('word2vec_model.bin')
print("Model Word2Vec disimpan ke 'word2vec_model.bin'")

### Perbedaan TF-IDF dan Word2Vec

| Aspek | TF-IDF | Word2Vec |
|-------|--------|----------|
| Dasar | Statistik frekuensi kata | Jaringan saraf tiruan |
| Makna Semantik | Tidak menangkap | Menangkap makna kontekstual |
| Dimensi | Sebesar jumlah fitur (5000) | Tetap (100 dimensi) |
| Nilai Vektor | Merepresentasikan kepentingan kata | Merepresentasikan makna dan konteks kata |
| Kelemahan | Tidak menangkap urutan kata | Membutuhkan data lebih banyak untuk melatih |

---
# Bagian 5 - Pemodelan Machine Learning

## Tujuan

Pada bagian ini, kita melatih dua algoritma machine learning untuk memprediksi rating (1-5 bintang) berdasarkan teks review. Setiap algoritma dilatih dengan dua representasi teks (TF-IDF dan Word2Vec) dan dilakukan tuning hyperparameter untuk mendapatkan performa terbaik.

## Algoritma yang Digunakan

### 1. Random Forest Classifier
Random Forest adalah algoritma ensemble yang membangun banyak pohon keputusan (decision tree) secara acak dan menggabungkan prediksinya. Algoritma ini dipilih karena:
- Tahan terhadap overfitting dibanding single decision tree
- Bekerja dengan baik pada data berdimensi tinggi (seperti TF-IDF)
- Memberikan informasi pentingnya setiap fitur

### 2. Support Vector Machine (LinearSVC)
SVM mencari hyperplane optimal yang memisahkan kelas-kelas data dengan margin terbesar. LinearSVC digunakan karena:
- Sangat efektif untuk klasifikasi teks berdimensi tinggi
- Efisien secara komputasi untuk dataset besar
- Bekerja sangat baik dengan representasi TF-IDF


In [ ]:
# ================================================================
# BAGIAN 5: PEMODELAN MACHINE LEARNING
# ================================================================

# Kamus untuk menyimpan semua model dan hasil evaluasinya
results = {}

# ================================================================
# ALGORITMA 1: RANDOM FOREST
# ================================================================
print("=" * 60)
print("RANDOM FOREST - HYPERPARAMETER TUNING")
print("=" * 60)

# Mendefinisikan ruang pencarian hyperparameter untuk Random Forest
# n_estimators: jumlah pohon dalam forest
# max_depth: kedalaman maksimum setiap pohon (None = tidak terbatas)
# min_samples_split: jumlah minimum sampel untuk memecah node internal
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

# --- Random Forest dengan TF-IDF ---
print("\nTuning Random Forest dengan TF-IDF...")
rf_tfidf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=rf_param_grid,
    n_iter=10,           # Jumlah kombinasi yang dicoba
    cv=3,                # 3-fold cross validation
    scoring='f1_weighted',
    random_state=SEED,
    n_jobs=-1
)

# Melatih model dengan data TF-IDF
rf_tfidf_search.fit(X_train_tfidf, y_train)

print(f"Parameter terbaik RF + TF-IDF: {rf_tfidf_search.best_params_}")
print(f"Skor CV terbaik: {rf_tfidf_search.best_score_:.4f}")

# Menyimpan model terbaik
best_rf_tfidf = rf_tfidf_search.best_estimator_
results['RF_TFIDF'] = {'model': best_rf_tfidf, 'X_test': X_test_tfidf}

# --- Random Forest dengan Word2Vec ---
print("\nTuning Random Forest dengan Word2Vec...")
rf_w2v_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=rf_param_grid,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    random_state=SEED,
    n_jobs=-1
)

# Melatih model dengan data Word2Vec
rf_w2v_search.fit(X_train_w2v, y_train)

print(f"Parameter terbaik RF + Word2Vec: {rf_w2v_search.best_params_}")
print(f"Skor CV terbaik: {rf_w2v_search.best_score_:.4f}")

best_rf_w2v = rf_w2v_search.best_estimator_
results['RF_W2V'] = {'model': best_rf_w2v, 'X_test': X_test_w2v}

In [ ]:
# ================================================================
# ALGORITMA 2: SUPPORT VECTOR MACHINE (LinearSVC)
# ================================================================
print("=" * 60)
print("SUPPORT VECTOR MACHINE - HYPERPARAMETER TUNING")
print("=" * 60)

# Mendefinisikan ruang pencarian hyperparameter untuk LinearSVC
# C: parameter regularisasi (semakin kecil = regularisasi lebih kuat)
# max_iter: jumlah maksimum iterasi untuk konvergensi
svm_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'max_iter': [1000, 2000, 3000]
}

# --- SVM dengan TF-IDF ---
print("\nTuning SVM dengan TF-IDF...")
svm_tfidf_search = GridSearchCV(
    LinearSVC(random_state=SEED),
    param_grid=svm_param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

svm_tfidf_search.fit(X_train_tfidf, y_train)
print(f"Parameter terbaik SVM + TF-IDF: {svm_tfidf_search.best_params_}")
print(f"Skor CV terbaik: {svm_tfidf_search.best_score_:.4f}")

best_svm_tfidf = svm_tfidf_search.best_estimator_
results['SVM_TFIDF'] = {'model': best_svm_tfidf, 'X_test': X_test_tfidf}

# --- SVM dengan Word2Vec ---
print("\nTuning SVM dengan Word2Vec...")
svm_w2v_search = GridSearchCV(
    LinearSVC(random_state=SEED),
    param_grid=svm_param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

svm_w2v_search.fit(X_train_w2v, y_train)
print(f"Parameter terbaik SVM + Word2Vec: {svm_w2v_search.best_params_}")
print(f"Skor CV terbaik: {svm_w2v_search.best_score_:.4f}")

best_svm_w2v = svm_w2v_search.best_estimator_
results['SVM_W2V'] = {'model': best_svm_w2v, 'X_test': X_test_w2v}

---
# Bagian 6 - Pemodelan Deep Learning

## Tujuan

Pada bagian ini, kita membangun model deep learning menggunakan arsitektur Bidirectional LSTM (Long Short-Term Memory) dengan representasi Word2Vec sebagai input.

## Mengapa Bidirectional LSTM?

LSTM dipilih karena kemampuannya menangkap dependensi jangka panjang dalam data sekuensial (teks). Versi Bidirectional berarti model memproses teks dari dua arah (kiri ke kanan dan kanan ke kiri), sehingga setiap kata memiliki konteks dari kata sebelum dan sesudahnya. Hal ini sangat berguna untuk memahami sentimen dalam kalimat yang konteksnya bergantung pada seluruh urutan kata.

## Arsitektur Model

1. **Input Layer**: Menerima vektor Word2Vec berdimensi 100
2. **Bidirectional LSTM Layer**: Memproses urutan vektor dari dua arah
3. **Dropout Layer**: Mencegah overfitting dengan menonaktifkan sebagian neuron secara acak
4. **Dense Layer**: Lapisan fully connected untuk representasi fitur
5. **Output Layer**: 5 neuron dengan aktivasi softmax untuk prediksi 5 kelas rating


In [ ]:
# ================================================================
# BAGIAN 6: PEMODELAN DEEP LEARNING (BIDIRECTIONAL LSTM)
# ================================================================

# Mengubah label rating menjadi format one-hot encoding untuk output softmax
# Rating 1-5 diubah menjadi indeks 0-4 terlebih dahulu
y_train_dl = to_categorical(y_train - 1, num_classes=5)
y_test_dl = to_categorical(y_test - 1, num_classes=5)

# Merestrukturisasi data untuk input LSTM
# LSTM membutuhkan input 3D: (sampel, langkah_waktu, fitur)
# Kita memperlakukan setiap vektor dokumen sebagai satu langkah waktu
X_train_dl = X_train_w2v.reshape(X_train_w2v.shape[0], 1, X_train_w2v.shape[1])
X_test_dl = X_test_w2v.reshape(X_test_w2v.shape[0], 1, X_test_w2v.shape[1])

print(f"Dimensi input Deep Learning training: {X_train_dl.shape}")
print(f"Dimensi input Deep Learning testing : {X_test_dl.shape}")

def build_lstm_model(lstm_units=128, dropout_rate=0.3, learning_rate=0.001):
    """
    Fungsi untuk membangun model Bidirectional LSTM.
    
    Parameter:
    - lstm_units: jumlah unit pada lapisan LSTM
    - dropout_rate: tingkat dropout untuk regularisasi
    - learning_rate: laju pembelajaran optimizer Adam
    """
    model = Sequential([
        # Lapisan Bidirectional LSTM memproses urutan dari dua arah
        Bidirectional(LSTM(lstm_units, return_sequences=False), input_shape=(1, 100)),
        # Dropout mencegah overfitting dengan mengabaikan sebagian neuron secara acak
        Dropout(dropout_rate),
        # Dense layer dengan aktivasi ReLU untuk representasi fitur non-linear
        Dense(64, activation='relu'),
        Dropout(dropout_rate / 2),
        # Output layer dengan softmax menghasilkan probabilitas untuk 5 kelas rating
        Dense(5, activation='softmax')
    ])
    
    # Kompilasi model dengan optimizer Adam dan loss categorical crossentropy
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# ================================================================
# TUNING HYPERPARAMETER DEEP LEARNING
# ================================================================
print("\nMelakukan tuning hyperparameter model Deep Learning...")

# Mendefinisikan kombinasi hyperparameter yang akan dicoba
# Hyperparameter 1: Jumlah unit LSTM
# Hyperparameter 2: Dropout rate
# Hyperparameter 3: Learning rate
hp_configs = [
    {'lstm_units': 64,  'dropout_rate': 0.2, 'learning_rate': 0.001},
    {'lstm_units': 128, 'dropout_rate': 0.3, 'learning_rate': 0.001},
    {'lstm_units': 128, 'dropout_rate': 0.5, 'learning_rate': 0.0005},
]

# Callback EarlyStopping untuk menghentikan pelatihan jika tidak ada peningkatan
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,         # Menunggu 3 epoch tanpa perbaikan sebelum berhenti
    restore_best_weights=True
)

best_val_acc = 0
best_dl_config = None
best_history = None

for i, config in enumerate(hp_configs):
    print(f"\nKonfigurasi {i+1}/{len(hp_configs)}: {config}")
    
    # Membangun model dengan konfigurasi saat ini
    model = build_lstm_model(**config)
    
    # Melatih model
    history = model.fit(
        X_train_dl, y_train_dl,
        epochs=10,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Mendapatkan akurasi validasi terbaik dari konfigurasi ini
    val_acc = max(history.history['val_accuracy'])
    print(f"  Akurasi validasi terbaik: {val_acc:.4f}")
    
    # Menyimpan konfigurasi terbaik
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_dl_config = config
        best_history = history
        best_dl_model = model

print(f"\nKonfigurasi terbaik: {best_dl_config}")
print(f"Akurasi validasi terbaik: {best_val_acc:.4f}")

In [ ]:
# --- Menampilkan Arsitektur Model Terbaik ---
print("Arsitektur model Deep Learning terbaik:")
best_dl_model.summary()

# --- Visualisasi Kurva Pelatihan ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafik Loss selama pelatihan
axes[0].plot(best_history.history['loss'], label='Training Loss', color='#1976d2', linewidth=2)
axes[0].plot(best_history.history['val_loss'], label='Validation Loss', color='#d32f2f', linewidth=2, linestyle='--')
axes[0].set_title('Training Loss vs Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafik Accuracy selama pelatihan
axes[1].plot(best_history.history['accuracy'], label='Training Accuracy', color='#388e3c', linewidth=2)
axes[1].plot(best_history.history['val_accuracy'], label='Validation Accuracy', color='#f57c00', linewidth=2, linestyle='--')
axes[1].set_title('Training Accuracy vs Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Kurva Pelatihan Model Bidirectional LSTM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('kurva_pelatihan_dl.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Bagian 7 - Evaluasi Performa Model

## Tujuan

Pada bagian ini, kita mengevaluasi dan membandingkan performa semua model yang telah dilatih menggunakan data test. Metrik yang digunakan:

- **Accuracy**: Proporsi prediksi yang benar dari seluruh prediksi
- **Precision**: Dari semua prediksi positif, berapa yang benar-benar positif
- **Recall**: Dari semua data yang sebenarnya positif, berapa yang berhasil diprediksi
- **F1-Score**: Rata-rata harmonik antara Precision dan Recall; metrik utama karena mempertimbangkan ketidakseimbangan kelas


In [ ]:
# ================================================================
# BAGIAN 7: EVALUASI PERFORMA MODEL
# ================================================================

def evaluate_model(model_name, y_true, y_pred, is_dl=False):
    """
    Fungsi untuk menghitung semua metrik evaluasi model.
    
    Parameter:
    - model_name: nama model untuk ditampilkan
    - y_true: label kelas sebenarnya
    - y_pred: label kelas hasil prediksi model
    - is_dl: True jika model adalah deep learning
    """
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    }

# Daftar untuk menyimpan semua hasil evaluasi
eval_results = []
all_predictions = {}

# ----------------------------------------------------------------
# Evaluasi model Machine Learning
# ----------------------------------------------------------------
ml_configs = [
    ('RF_TFIDF',   'Random Forest',  'TF-IDF',   best_rf_tfidf,  X_test_tfidf),
    ('RF_W2V',     'Random Forest',  'Word2Vec',  best_rf_w2v,    X_test_w2v),
    ('SVM_TFIDF',  'SVM',            'TF-IDF',   best_svm_tfidf, X_test_tfidf),
    ('SVM_W2V',    'SVM',            'Word2Vec',  best_svm_w2v,   X_test_w2v),
]

for key, algo, rep, model, X_test in ml_configs:
    # Membuat prediksi menggunakan model yang sudah dilatih
    y_pred = model.predict(X_test)
    all_predictions[key] = y_pred
    
    metrics = evaluate_model(f"{algo} + {rep}", y_test, y_pred)
    metrics['Text Representation'] = rep
    eval_results.append(metrics)
    
    print(f"\nClassification Report - {algo} + {rep}:")
    print(classification_report(y_test, y_pred, zero_division=0))

# ----------------------------------------------------------------
# Evaluasi model Deep Learning
# ----------------------------------------------------------------
print("\nEvaluasi Deep Learning...")
# Mendapatkan probabilitas prediksi dari output softmax
y_pred_dl_prob = best_dl_model.predict(X_test_dl)
# Mengubah probabilitas menjadi kelas prediksi (indeks tertinggi + 1)
y_pred_dl = np.argmax(y_pred_dl_prob, axis=1) + 1
all_predictions['BiLSTM_W2V'] = y_pred_dl

dl_metrics = evaluate_model('Bidirectional LSTM + Word2Vec', y_test, y_pred_dl)
dl_metrics['Text Representation'] = 'Word2Vec'
eval_results.append(dl_metrics)

print(f"\nClassification Report - Bidirectional LSTM + Word2Vec:")
print(classification_report(y_test, y_pred_dl, zero_division=0))

In [ ]:
# --- Tabel Ringkasan Hasil Evaluasi ---
summary_df = pd.DataFrame(eval_results)[['Model', 'Text Representation', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
summary_df = summary_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

print("=" * 80)
print("TABEL RINGKASAN PERFORMA SEMUA MODEL (Data Asli)")
print("=" * 80)
print(summary_df.to_string(index=False))

In [ ]:
# --- Visualisasi Confusion Matrix untuk Setiap Model ---
model_keys = ['RF_TFIDF', 'RF_W2V', 'SVM_TFIDF', 'SVM_W2V', 'BiLSTM_W2V']
model_labels = ['RF + TF-IDF', 'RF + Word2Vec', 'SVM + TF-IDF', 'SVM + Word2Vec', 'BiLSTM + Word2Vec']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, (key, label) in enumerate(zip(model_keys, model_labels)):
    # Membuat confusion matrix
    cm = confusion_matrix(y_test, all_predictions[key])
    
    # Menampilkan confusion matrix sebagai heatmap
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=[1, 2, 3, 4, 5],
        yticklabels=[1, 2, 3, 4, 5],
        ax=axes[i],
        linewidths=0.5
    )
    axes[i].set_title(f'Confusion Matrix\n{label}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Predicted Rating')
    axes[i].set_ylabel('Actual Rating')

axes[5].axis('off')
plt.suptitle('Confusion Matrix Semua Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_semua_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualisasi Perbandingan F1-Score Semua Model ---
fig, ax = plt.subplots(figsize=(12, 6))

# Memisahkan model berdasarkan text representation untuk pengelompokan
tfidf_models = summary_df[summary_df['Text Representation'] == 'TF-IDF']
w2v_models = summary_df[summary_df['Text Representation'] == 'Word2Vec']

x = np.arange(len(summary_df))
bar_colors_map = {'TF-IDF': '#1976d2', 'Word2Vec': '#388e3c'}
bar_colors_list = [bar_colors_map[r] for r in summary_df['Text Representation']]

bars = ax.bar(x, summary_df['F1-Score'], color=bar_colors_list, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(summary_df['Model'], rotation=15, ha='right', fontsize=9)
ax.set_title('Perbandingan F1-Score Semua Model', fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Score (Weighted)')
ax.set_ylim(0, 1.05)

# Menambahkan nilai di atas setiap bar
for bar, val in zip(bars, summary_df['F1-Score']):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Menambahkan legenda untuk warna representasi teks
tfidf_patch = mpatches.Patch(color='#1976d2', label='TF-IDF')
w2v_patch = mpatches.Patch(color='#388e3c', label='Word2Vec')
ax.legend(handles=[tfidf_patch, w2v_patch], title='Text Representation')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('perbandingan_f1_score.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Bagian 8 - Penanganan Imbalance Data

## Tujuan

Dari EDA pada Bagian 2, kita telah melihat bahwa dataset tidak seimbang (imbalanced), di mana rating 5 bintang mendominasi. Kondisi ini dapat menyebabkan model bias terhadap kelas mayoritas dan mengabaikan kelas minoritas.

## Teknik Penanganan

### Teknik 1: SMOTE (Synthetic Minority Over-sampling Technique)
SMOTE menghasilkan data sintetis baru untuk kelas minoritas dengan cara menginterpolasi antara sampel-sampel yang ada. Ini efektif untuk menambah data kelas minoritas tanpa sekadar menduplikasi.

### Teknik 2: Class Weight Adjustment
Memberikan bobot yang lebih tinggi pada kelas minoritas saat proses pelatihan. Model akan memberi penalti lebih besar ketika salah mengklasifikasikan kelas minoritas, sehingga lebih memperhatikannya.


In [ ]:
# ================================================================
# BAGIAN 8: PENANGANAN IMBALANCE DATA
# ================================================================

# --- 8.1 Memeriksa dan Memvisualisasikan Imbalance Data ---
print("Distribusi kelas pada data training (sebelum penanganan):")
train_dist = pd.Series(y_train).value_counts().sort_index()
print(train_dist)

# Menghitung rasio ketidakseimbangan
max_class = train_dist.max()
min_class = train_dist.min()
imbalance_ratio = max_class / min_class
print(f"\nRasio ketidakseimbangan (max/min): {imbalance_ratio:.2f}x")
print("Dataset ini dikategorikan imbalanced jika rasio > 1.5")

# Visualisasi distribusi sebelum penanganan
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot distribusi asli
axes[0].bar(train_dist.index, train_dist.values, color=colors, edgecolor='black')
axes[0].set_title('Distribusi Kelas - Data Asli', fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate(train_dist.values):
    axes[0].text(i + 1, v + 2, str(v), ha='center', fontweight='bold', fontsize=9)

# ================================================================
# TEKNIK 1: SMOTE - Diterapkan pada TF-IDF
# ================================================================
print("\nMenerapkan SMOTE pada data TF-IDF...")

smote = SMOTE(random_state=SEED)
# Mengubah matriks sparse TF-IDF menjadi dense array untuk SMOTE
X_train_tfidf_dense = X_train_tfidf.toarray()
X_train_tfidf_smote, y_train_smote = smote.fit_resample(X_train_tfidf_dense, y_train)

print("Distribusi setelah SMOTE:")
smote_dist = pd.Series(y_train_smote).value_counts().sort_index()
print(smote_dist)

# Plot setelah SMOTE
axes[1].bar(smote_dist.index, smote_dist.values, color=colors, edgecolor='black')
axes[1].set_title('Distribusi Kelas - Setelah SMOTE', fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Jumlah Sampel')
for i, v in enumerate(smote_dist.values):
    axes[1].text(i + 1, v + 2, str(v), ha='center', fontweight='bold', fontsize=9)

# ================================================================
# TEKNIK 2: CLASS WEIGHT - Diterapkan pada Word2Vec
# ================================================================
# Menghitung bobot kelas secara otomatis berdasarkan frekuensi
# Kelas dengan sampel lebih sedikit akan mendapat bobot lebih besar
classes = np.unique(y_train)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights_array))
print("\nBobot kelas untuk penanganan imbalance:")
for cls, weight in class_weight_dict.items():
    print(f"  Rating {cls}: bobot = {weight:.4f}")

# Visualisasi bobot kelas
axes[2].bar(list(class_weight_dict.keys()), list(class_weight_dict.values()), 
            color=colors, edgecolor='black')
axes[2].set_title('Class Weight per Rating', fontweight='bold')
axes[2].set_xlabel('Rating')
axes[2].set_ylabel('Bobot Kelas')

plt.suptitle('Penanganan Imbalance Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_handling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# MELATIH ULANG SEMUA MODEL DENGAN DATA YANG SUDAH DIBALANCE
# ================================================================
balanced_eval_results = []

# --- Random Forest + TF-IDF dengan SMOTE ---
print("Melatih Random Forest + TF-IDF (SMOTE)...")
rf_tfidf_smote = RandomForestClassifier(
    **rf_tfidf_search.best_params_,
    random_state=SEED, n_jobs=-1
)
rf_tfidf_smote.fit(X_train_tfidf_smote, y_train_smote)
y_pred = rf_tfidf_smote.predict(X_test_tfidf)
m = evaluate_model('Random Forest + TF-IDF', y_test, y_pred)
m['Text Representation'] = 'TF-IDF'
m['Kondisi Data'] = 'SMOTE'
balanced_eval_results.append(m)
print(f"  F1-Score: {m['F1-Score']}")

# --- SVM + TF-IDF dengan SMOTE ---
print("Melatih SVM + TF-IDF (SMOTE)...")
svm_tfidf_smote = LinearSVC(
    **svm_tfidf_search.best_params_,
    random_state=SEED
)
svm_tfidf_smote.fit(X_train_tfidf_smote, y_train_smote)
y_pred = svm_tfidf_smote.predict(X_test_tfidf)
m = evaluate_model('SVM + TF-IDF', y_test, y_pred)
m['Text Representation'] = 'TF-IDF'
m['Kondisi Data'] = 'SMOTE'
balanced_eval_results.append(m)
print(f"  F1-Score: {m['F1-Score']}")

# --- Random Forest + Word2Vec dengan Class Weight ---
print("Melatih Random Forest + Word2Vec (Class Weight)...")
rf_w2v_cw = RandomForestClassifier(
    **rf_w2v_search.best_params_,
    class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
rf_w2v_cw.fit(X_train_w2v, y_train)
y_pred = rf_w2v_cw.predict(X_test_w2v)
m = evaluate_model('Random Forest + Word2Vec', y_test, y_pred)
m['Text Representation'] = 'Word2Vec'
m['Kondisi Data'] = 'Class Weight'
balanced_eval_results.append(m)
print(f"  F1-Score: {m['F1-Score']}")

# --- SVM + Word2Vec dengan Class Weight ---
print("Melatih SVM + Word2Vec (Class Weight)...")
svm_w2v_cw = LinearSVC(
    **svm_w2v_search.best_params_,
    class_weight='balanced',
    random_state=SEED
)
svm_w2v_cw.fit(X_train_w2v, y_train)
y_pred = svm_w2v_cw.predict(X_test_w2v)
m = evaluate_model('SVM + Word2Vec', y_test, y_pred)
m['Text Representation'] = 'Word2Vec'
m['Kondisi Data'] = 'Class Weight'
balanced_eval_results.append(m)
print(f"  F1-Score: {m['F1-Score']}")

# --- Deep Learning dengan Class Weight ---
print("Melatih Deep Learning (Class Weight)...")
# Mengonversi kunci class_weight_dict agar sesuai dengan indeks (0-4)
dl_class_weight = {int(k) - 1: v for k, v in class_weight_dict.items()}

dl_model_balanced = build_lstm_model(**best_dl_config)
dl_model_balanced.fit(
    X_train_dl, y_train_dl,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=dl_class_weight,  # Menerapkan bobot kelas saat pelatihan
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=0
)

y_pred_dl_bal = np.argmax(dl_model_balanced.predict(X_test_dl), axis=1) + 1
m = evaluate_model('Bidirectional LSTM + Word2Vec', y_test, y_pred_dl_bal)
m['Text Representation'] = 'Word2Vec'
m['Kondisi Data'] = 'Class Weight'
balanced_eval_results.append(m)
print(f"  F1-Score: {m['F1-Score']}")

In [ ]:
# ================================================================
# TABEL PERBANDINGAN SEBELUM DAN SESUDAH PENANGANAN IMBALANCE
# ================================================================

# Menambahkan kolom kondisi data pada hasil sebelumnya
original_results = []
for r in eval_results:
    r_copy = r.copy()
    r_copy['Kondisi Data'] = 'Asli'
    original_results.append(r_copy)

# Menggabungkan semua hasil evaluasi
combined_df = pd.DataFrame(original_results + balanced_eval_results)
combined_df = combined_df[['Model', 'Text Representation', 'Kondisi Data', 
                            'Accuracy', 'Precision', 'Recall', 'F1-Score']]
combined_df = combined_df.sort_values(['Model', 'Kondisi Data']).reset_index(drop=True)

print("=" * 100)
print("TABEL PERBANDINGAN PERFORMA SEBELUM DAN SESUDAH PENANGANAN IMBALANCE")
print("=" * 100)
print(combined_df.to_string(index=False))

In [ ]:
# --- Visualisasi Perbandingan Sebelum dan Sesudah Imbalance Handling ---
# Mempersiapkan data untuk visualisasi
model_names_unique = combined_df['Model'].unique()

# Mengambil F1-Score kondisi Asli dan kondisi setelah penanganan
asli_f1 = []
balanced_f1 = []
model_labels_short = []

for model_name in model_names_unique:
    sub_df = combined_df[combined_df['Model'] == model_name]
    asli_row = sub_df[sub_df['Kondisi Data'] == 'Asli']
    balanced_row = sub_df[sub_df['Kondisi Data'] != 'Asli']
    
    if not asli_row.empty and not balanced_row.empty:
        asli_f1.append(asli_row['F1-Score'].values[0])
        balanced_f1.append(balanced_row['F1-Score'].values[0])
        model_labels_short.append(model_name.replace(' + ', '\n+'))

x = np.arange(len(model_labels_short))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, asli_f1, width, label='Data Asli (Imbalanced)', 
               color='#ef5350', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, balanced_f1, width, label='Setelah Penanganan Imbalance',
               color='#42a5f5', edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(model_labels_short, fontsize=8)
ax.set_title('Perbandingan F1-Score Sebelum dan Sesudah Penanganan Imbalance', 
             fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Score (Weighted)')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Menambahkan nilai di atas setiap bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=7, rotation=45)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig('perbandingan_imbalance_handling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# TABEL RINGKASAN AKHIR SEMUA EKSPERIMEN
# ================================================================
print("=" * 100)
print("TABEL RINGKASAN AKHIR - SEMUA MODEL DAN KONDISI DATA")
print("=" * 100)
print(combined_df.to_string(index=False))

# Menyimpan hasil akhir ke file CSV untuk dokumentasi
combined_df.to_csv('hasil_evaluasi_lengkap.csv', index=False)
print("\nHasil evaluasi lengkap disimpan ke 'hasil_evaluasi_lengkap.csv'")
print("\nProyek analisis NLP review Wuthering Waves selesai.")